# HW12 - Deep Reinforcement Learning
## Lunar Lander with Actor-Critic

演算法：Actor-Critic（REINFORCE with Baseline）
- **Actor**：輸出每個 action 的機率分佈
- **Critic**：估計當前 state 的 value V(s)，作為 baseline
- **Advantage**：A_t = R_t - V(s_t)，降低 Policy Gradient 的 variance

In [35]:
# 安裝必要套件
# 第一次「全部執行」會安裝套件並自動重啟 kernel
# 第二次「全部執行」偵測到套件已存在，直接跳過
try:
    import pyvirtualdisplay
    print('套件已安裝，繼續執行。')
except ImportError:
    import os
    !apt update -qq
    !apt install -y -qq python3-opengl xvfb swig
    # 改用 gymnasium：gym 的官方後繼版，LunarLander 環境相同，支援現代 pip
    !pip install gymnasium[box2d] pyvirtualdisplay tqdm -q
    print('安裝完成，正在重啟 kernel，請稍後再次點「全部執行」...')
    os.kill(os.getpid(), 9)

套件已安裝，繼續執行。


In [36]:
# 啟動虛擬顯示器（Colab 沒有螢幕，gym render 需要這個）
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [37]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython import display

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
from tqdm.notebook import tqdm
import gymnasium as gym  # gymnasium 是 gym 的現代後繼版
import random

print('PyTorch 版本:', torch.__version__)
print('GPU 可用:', torch.cuda.is_available())

PyTorch 版本: 2.10.0+cu128
GPU 可用: True


## 固定隨機種子與環境初始化

In [38]:
seed = 543

def fix(env, seed):
    # gymnasium 的 reset() 用 seed 參數傳入，不再有 env.seed()
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

env = gym.make('LunarLander-v3')  # gymnasium 新版棄用 v2，改用 v3
fix(env, seed)

print('觀測空間維度:', env.observation_space.shape)  # (8,)
print('動作空間大小:', env.action_space.n)            # 4

觀測空間維度: (8,)
動作空間大小: 4


## Actor-Critic 網路架構

共用特徵提取層（shared layers）提取 state 的特徵，再分叉成兩個 head：
- **Actor head**：輸出 4 個 action 的機率，用 Softmax 歸一化
- **Critic head**：輸出純量 V(s)，代表此 state 預期能拿到的總 reward

In [39]:
class ActorCriticNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # 加大網路容量：64 -> 128
        self.shared = nn.Sequential(
            nn.Linear(8, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
        )
        self.actor_head = nn.Linear(128, 4)
        self.critic_head = nn.Linear(128, 1)

    def forward(self, state):
        features = self.shared(state)
        action_probs = F.softmax(self.actor_head(features), dim=-1)
        state_value = self.critic_head(features)
        return action_probs, state_value

## Actor-Critic Agent

### 核心概念

**1. 折扣累積 Reward**：
$$R_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots = \sum_{i=t}^{T} \gamma^{i-t} r_i$$

**2. Advantage（優勢函數）**：
$$A_t = R_t - V(s_t)$$
- $A_t > 0$：這個 action 比 Critic 預期的還好 → 增加機率
- $A_t < 0$：這個 action 比 Critic 預期的還差 → 減少機率

**3. Loss 函數**：
- Actor loss = $-\log \pi(a_t|s_t) \cdot A_t$
- Critic loss = $(V(s_t) - R_t)^2$
- Total loss = Actor loss + 0.5 × Critic loss

In [40]:
class ActorCriticAgent:
    def __init__(self, network):
        self.network = network
        self.optimizer = optim.Adam(self.network.parameters(), lr=3e-4)
        self.gamma = 0.99
        self.entropy_coef = 0.01  # entropy bonus 鼓勵探索

    def sample(self, state):
        state_tensor = torch.FloatTensor(state)
        action_probs, state_value = self.network(state_tensor)
        dist = Categorical(action_probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()  # 用於 entropy bonus
        return action.item(), log_prob, state_value, entropy

    def discount_rewards(self, rewards):
        """計算單一 episode 的折扣累積 reward R_t"""
        discounted = []
        running = 0
        for r in reversed(rewards):
            running = r + self.gamma * running
            discounted.insert(0, running)
        return torch.FloatTensor(discounted)

    def learn(self, episodes):
        """
        episodes: list of (log_probs, state_values, rewards, entropies)
        每個 episode 分開計算 returns，再合併更新，避免跨 episode 污染
        """
        all_log_probs = []
        all_values = []
        all_returns = []
        all_entropies = []

        for log_probs, state_values, rewards, entropies in episodes:
            returns = self.discount_rewards(rewards)
            # 每個 episode 內部做標準化
            returns = (returns - returns.mean()) / (returns.std() + 1e-9)
            all_returns.append(returns)
            all_log_probs.extend(log_probs)
            all_values.extend(state_values)
            all_entropies.extend(entropies)

        returns = torch.cat(all_returns)
        log_probs = torch.stack(all_log_probs)
        values = torch.stack(all_values).squeeze()
        entropies = torch.stack(all_entropies)

        advantages = returns - values.detach()

        actor_loss = (-log_probs * advantages).mean()
        critic_loss = F.mse_loss(values, returns)
        # entropy bonus：讓 policy 不要太快收斂，保持探索
        entropy_loss = -entropies.mean()

        loss = actor_loss + 0.5 * critic_loss + self.entropy_coef * entropy_loss

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.network.parameters(), max_norm=0.5)
        self.optimizer.step()

    def save(self, path):
        torch.save({
            'network': self.network.state_dict(),
            'optimizer': self.optimizer.state_dict()
        }, path)

    def load(self, path):
        checkpoint = torch.load(path)
        self.network.load_state_dict(checkpoint['network'])
        self.optimizer.load_state_dict(checkpoint['optimizer'])

## 訓練迴圈

In [ ]:
network = ActorCriticNetwork()
agent = ActorCriticAgent(network)

EPISODE_PER_BATCH = 10   # 5 -> 10，梯度估計更穩定
NUM_BATCH = 3000          # 給更多訓練時間(1500->3000)

avg_total_rewards = []

agent.network.train()
prg_bar = tqdm(range(NUM_BATCH))

for batch in prg_bar:
    episodes = []
    total_rewards = []

    for episode in range(EPISODE_PER_BATCH):
        if batch == 0 and episode == 0:
            state, _ = env.reset(seed=seed)
        else:
            state, _ = env.reset()

        total_reward = 0
        ep_log_probs, ep_values, ep_rewards, ep_entropies = [], [], [], []

        while True:
            action, log_prob, state_value, entropy = agent.sample(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            ep_log_probs.append(log_prob)
            ep_values.append(state_value)
            ep_rewards.append(reward)
            ep_entropies.append(entropy)

            state = next_state
            total_reward += reward

            if done:
                total_rewards.append(total_reward)
                break

        episodes.append((ep_log_probs, ep_values, ep_rewards, ep_entropies))

    agent.learn(episodes)

    avg_reward = sum(total_rewards) / len(total_rewards)
    avg_total_rewards.append(avg_reward)
    prg_bar.set_description(f"Avg Reward: {avg_reward:6.1f}")

agent.save('actor_critic.pth')
print('訓練完成，模型已儲存。')

  0%|          | 0/3000 [00:00<?, ?it/s]

## 訓練曲線

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(avg_total_rewards, alpha=0.4, label='每 batch 平均 Reward')

# 移動平均平滑曲線
window = 20
if len(avg_total_rewards) >= window:
    smoothed = np.convolve(avg_total_rewards, np.ones(window) / window, mode='valid')
    plt.plot(range(window - 1, len(avg_total_rewards)), smoothed,
             color='red', label=f'{window}-batch 移動平均')

plt.axhline(y=200, color='green', linestyle='--', alpha=0.5, label='目標 200')
plt.xlabel('Batch')
plt.ylabel('Total Reward')
plt.title('Actor-Critic on LunarLander-v2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'最後 50 個 batch 的平均 Reward: {np.mean(avg_total_rewards[-50:]):.1f}')

## 產生 Action List（供 JudgeBoi 評分）

用訓練好的模型跑 5 個 episode，以 greedy 方式選 action（取機率最高的），存成 `action_list.npy` 上傳。

In [ ]:
# agent.load('actor_critic.pth')  # 若要從已儲存的模型繼續，取消此行註解

agent.network.eval()

NUM_OF_TEST = 5
action_list = []
total_scores = []

for episode in range(NUM_OF_TEST):
    state, _ = env.reset(seed=seed)  # gymnasium: reset() 回傳 (obs, info)
    total_score = 0
    ep_actions = []

    while True:
        state_tensor = torch.FloatTensor(state)
        with torch.no_grad():
            action_probs, _ = agent.network(state_tensor)
        action = action_probs.argmax().item()

        ep_actions.append(action)
        state, reward, terminated, truncated, _ = env.step(action)
        total_score += reward

        if terminated or truncated:
            total_scores.append(total_score)
            break

    action_list.append(ep_actions)

print('Action list looks like :', action_list)
print('Action list\'s shape looks like :', np.shape(action_list))
print(f'\n5 個 episode 平均分數: {np.mean(total_scores):.1f}')

In [ ]:
# 儲存 action list，上傳到 JudgeBoi
np.save('action_list.npy', action_list)
print('action_list.npy 已儲存。')

## 視覺化 Agent 表現（選用）

In [ ]:
# gymnasium 需要在 make 時指定 render_mode，建立獨立的渲染用 env
render_env = gym.make('LunarLander-v3', render_mode='rgb_array')

agent.network.eval()
state, _ = render_env.reset(seed=seed)
img = plt.imshow(render_env.render())
plt.axis('off')

while True:
    state_tensor = torch.FloatTensor(state)
    with torch.no_grad():
        action_probs, _ = agent.network(state_tensor)
    action = action_probs.argmax().item()

    state, reward, terminated, truncated, _ = render_env.step(action)
    img.set_data(render_env.render())
    display.display(plt.gcf())
    display.clear_output(wait=True)

    if terminated or truncated:
        break

render_env.close()
print('完成一局。')